# SQL y Bases de Datos Relacionales con Python

## 🎯 Objetivos de Aprendizaje
- Dominar los fundamentos de SQL para analistas: `SELECT`, `WHERE`, `JOIN`, `GROUP BY` y `HAVING`.
- Conectar y ejecutar consultas SQL en bases de datos embebidas (`sqlite3`) y clientes cliente-servidor (PostgreSQL).
- Integrar SQL y Pandas bidireccionalmente con `pd.read_sql_query()` y `df.to_sql()`.
- Implementar el uso de parámetros de consulta para prevenir inyecciones SQL.
- Conocer el rol de SQLAlchemy como motor ORM y abstracción de conexión en Python 3.12+.

## 🌉 Puente Pedagógico: SQL y Pandas como Aliados

### ¿Por qué SQL es imprescindible?
En empresas y producción, los datos rara vez viven en archivos CSV locales. Viven en bases de datos relacionales empresariales (PostgreSQL, MySQL, Snowflake, BigQuery). Filtrar y agregar datos en el motor de base de datos antes de cargarlos a memoria en Python ahorra gigabytes de RAM.

### Analogía
- **SQL**: El montacargas de un gran almacén logístico que filtra y te trae solo las 100 cajas que necesitas.
- **Pandas**: Tu mesa de trabajo de laboratorio donde inspeccionas minuciosamente esas 100 cajas con algoritmos sofisticados.

### Diagrama del Flujo de Consulta
```
   [ Base de Datos Relacional ] (Gigas / Teras de Datos)
                |
                |  <--- Consulta SQL (SELECT ... WHERE ... JOIN)
                v
     [ Motor de Base de Datos ] (Filtra en disco y calcula agregaciones)
                |
                |  ---> Retorna solo el subconjunto relevante
                v
      [ Python / Pandas ] (DataFrame listo para ML / Visualización en RAM)
```

In [ ]:
import sqlite3
import pandas as pd

# Conexión a una base de datos en memoria (ideal para pruebas y pipelines ligeros)
conn = sqlite3.connect(":memory:")

# Creación y llenado de tablas relacionales de ejemplo
conn.executescript("""
CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    ciudad TEXT NOT NULL
);

CREATE TABLE compras (
    id_compra INTEGER PRIMARY KEY,
    id_cliente INTEGER,
    monto REAL NOT NULL,
    fecha TEXT NOT NULL,
    FOREIGN KEY(id_cliente) REFERENCES clientes(id_cliente)
);

INSERT INTO clientes VALUES (1, 'Mariana', 'CDMX'), (2, 'Jorge', 'Guadalajara'), (3, 'Sofia', 'Monterrey');
INSERT INTO compras VALUES (101, 1, 1500.50, '2026-01-10'), (102, 1, 320.00, '2026-02-14'), (103, 2, 2800.00, '2026-02-20');
""")
conn.commit()
print("Base de datos SQLite en memoria inicializada exitosamente.")

## 1. Consultas y Cruces (JOINs) con `pd.read_sql_query`

Podemos ejecutar consultas complejas con `JOIN` y agregaciones directas, cargando el resultado directamente en un DataFrame de Pandas.

In [ ]:
query = """
SELECT 
    c.nombre,
    c.ciudad,
    COUNT(p.id_compra) AS total_compras,
    COALESCE(SUM(p.monto), 0) AS gasto_total
FROM clientes c
LEFT JOIN compras p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY gasto_total DESC;
"""

df_reporte = pd.read_sql_query(query, conn)
display(df_reporte)

## 2. Consultas Parametrizadas: Prevención de Inyección SQL

> [!WARNING]
> **Nunca concatenes strings** (`f"SELECT ... WHERE id = {user_input}"`) para armar consultas SQL. Siempre utiliza placeholders (`?` en sqlite3, `:param` en SQLAlchemy).

In [ ]:
ciudad_filtro = "Guadalajara"
query_segura = "SELECT * FROM clientes WHERE ciudad = ?"
df_filtrado = pd.read_sql_query(query_segura, conn, params=[ciudad_filtro])
display(df_filtrado)

## 📝 Ejercicios Prácticos

### Ejercicio 1 (Guiado): Escribir un DataFrame en SQL
Crea un nuevo DataFrame en Pandas y guárdalo como tabla `productos` usando `.to_sql()`.

In [ ]:
df_productos = pd.DataFrame({
    "id_prod": [1, 2, 3],
    "nombre": ["Monitor 27''", "Mouse Ergonómico", "Silla Gaming"],
    "stock": [15, 50, 8]
})

df_productos.to_sql("productos", conn, if_exists="replace", index=False)
verificacion = pd.read_sql_query("SELECT * FROM productos WHERE stock < 20", conn)
display(verificacion)

### Ejercicio 2 (Independiente): Cálculo de Ticket Promedio
Escribe una consulta SQL que calcule el monto promedio por compra de todos los clientes que tengan al menos una compra.

In [ ]:
# Solución:
query_promedio = """
SELECT 
    AVG(monto) AS ticket_promedio,
    MAX(monto) AS compra_maxima
FROM compras;
"""
display(pd.read_sql_query(query_promedio, conn))

## 📋 Tabla de Comandos y Resumen

| Comando SQL | Propósito | Equivalente en Pandas |
|:---|:---|:---|
| `SELECT col1, col2` | Proyección de columnas | `df[["col1", "col2"]]` |
| `WHERE condicion` | Filtro de filas | `df.query("condicion")` / `df[df[...] ]` |
| `JOIN ... ON ...` | Cruce relacional | `pd.merge(..., how=...)` |
| `GROUP BY ...` | Agrupamiento analítico | `df.groupby(...).agg(...)` |
| `ORDER BY ... DESC` | Ordenamiento de registros | `df.sort_values(ascending=False)` |

---
### 💡 Conclusión
Dominar SQL junto con Pandas te permite aprovechar lo mejor de ambos mundos: la capacidad de cálculo y agregación masiva en el motor de almacenamiento relacional, junto con la flexibilidad computacional de Python para modelado y visualización avanzada.